<a href="https://colab.research.google.com/github/jilliangreene/sta_554_assignments/blob/main/HW7/Greene_STA554_HW7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# STA 554 HW 7
Jillian Greene
03/31/2026

In this assignment, we are using a UCI ML Repo dataset to model `alcohol` (MLR) or type of wine (LR). Data downloaded from [here](https://archive.ics.uci.edu/dataset/186/wine+quality)

In [40]:
# Import packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV
from sklearn.linear_model import LinearRegression, LassoCV, Ridge, ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [34]:
# Read in red wine csv
reds = pd.read_csv('winequality-red.csv', sep=';')
# Add wine_type col
reds['wine_type'] = 'red'
# print length
print(len(reds))

# White wine
whites = pd.read_csv('winequality-white.csv', sep=';')
whites['wine_type'] = 'white'
print(len(whites))

# Combine the two dfs
df = pd.concat([reds, whites])
print(len(df))

1599
4898
6497


Now the two datasets have been read in, added a column for type of wine, and binded together. I've used the `print(len())` tool to ensure the binding happened as expected.

In [4]:
# Use train_test_splt() to do stratified split with equal parts by wine_type
train, test = train_test_split(df, test_size=0.7, stratify=df['wine_type'], random_state=120301)
# Print check
print(len(train[train['wine_type'] == 'white']) / len(test[test['wine_type'] == 'white']))
print(len(train[train['wine_type'] == 'red']) / len(test[test['wine_type'] == 'red']))

0.4284047827354914
0.4289544235924933


In [15]:
# Set x and y for train-test split
x_train = train.drop(['alcohol', 'wine_type'], axis=1)
y_train = train['alcohol']

x_test = test.drop(['alcohol', 'wine_type'], axis=1)
y_test = test['alcohol']

Now that the data is split up proportionally equal, I can begin the regression task from HW 7, beginning with 4 MLRs

In [16]:
# MLR 1 with interaction terms
model1 = Pipeline([
    ("scaler", StandardScaler()),
    ("interactions", PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)),
    ("lr", LinearRegression())])

In [17]:
# MLR 2 (standard)
model2 = Pipeline([("scaler", StandardScaler()),
    ("lr", LinearRegression())])

In [18]:
# 2-deg polynomial
model3 = Pipeline([("scaler", StandardScaler()),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("lr", LinearRegression())])

In [19]:
# 4-degree polynomial
model4 = Pipeline([("scaler", StandardScaler()),
    ("poly", PolynomialFeatures(degree=4, include_bias=False)),
    ("lr", LinearRegression())])

In [21]:
# Run models and compare CV scores
models = {"Interaction": model1,
    "Standard": model2,
    "Poly2": model3,
    "Poly4": model4}

scoring = {"rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2"}

results = {}

for name, model in models.items():
    scores = cross_validate(
        model, x_train, y_train,
        cv=5,
        scoring=scoring)

    results[name] = {"RMSE": -np.mean(scores["test_rmse"]),
        "MAE": -np.mean(scores["test_mae"]),
        "R2": np.mean(scores["test_r2"])}

# Print scores
for name, metrics in results.items():
    print(f"\n{name}")
    print(f"RMSE: {metrics['RMSE']:.4f}")
    print(f"MAE:  {metrics['MAE']:.4f}")
    print(f"R2:   {metrics['R2']:.4f}")


Interaction
RMSE: 0.5160
MAE:  0.3714
R2:   0.8094

Standard
RMSE: 0.5175
MAE:  0.3933
R2:   0.8096

Poly2
RMSE: 0.4991
MAE:  0.3566
R2:   0.8206

Poly4
RMSE: 47.0902
MAE:  6.9141
R2:   -2417.1671


The best MLR model was the one that used 2-degree polynomial terms (the 4-degree model was significantly bad). The next 3 models require a set of predictors, so I'll filter those down and try the models.

In [22]:
x_test.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,quality
3010,6.7,0.25,0.31,1.35,0.061,30.5,218.0,0.99388,3.16,0.53,5
134,6.8,0.27,0.22,8.10,0.034,55.0,203.0,0.99610,3.19,0.52,5
1161,8.8,0.45,0.43,1.40,0.076,12.0,21.0,0.99551,3.21,0.75,6
3990,7.3,0.23,0.41,14.60,0.048,73.0,223.0,0.99863,3.16,0.71,6
1457,7.6,0.49,0.33,1.90,0.074,27.0,85.0,0.99706,3.41,0.58,5


In [26]:
# print x_train col names
x_train.columns

Index(['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
       'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
       'pH', 'sulphates', 'quality'],
      dtype='object')

In [27]:
# 6-predictor models
x_train6 = x_train[['citric acid', 'residual sugar', 'fixed acidity', 'sulphates', 'pH', 'chlorides']]

x_test6 = x_test[['citric acid', 'residual sugar', 'fixed acidity', 'sulphates', 'pH', 'chlorides']]

In [32]:
# LASSO model with > 5 predictors, with CV tuning
lasso_model = Pipeline([("scaler", StandardScaler()),
    ("lasso", LassoCV(cv = 10))])

lasso_model.fit(x_train6, y_train)

best_alpha = lasso_model.named_steps["lasso"].alpha_
print("Best alpha:", best_alpha)

lasso_scores = cross_validate(lasso_model,
    x_train6,
    y_train,
    cv=5,
    scoring=scoring)

lasso_results = {"RMSE": -np.mean(lasso_scores["test_rmse"]),
    "MAE": -np.mean(lasso_scores["test_mae"]),
    "R2":  np.mean(lasso_scores["test_r2"])}

results["LASSO"] = lasso_results

# Print scores
for name, metrics in results.items():
    print(f"\n{name}")
    print(f"RMSE: {metrics['RMSE']:.4f}")
    print(f"MAE:  {metrics['MAE']:.4f}")
    print(f"R2:   {metrics['R2']:.4f}")

Best alpha: 0.006965921073118002

Interaction
RMSE: 0.5160
MAE:  0.3714
R2:   0.8094

Standard
RMSE: 0.5175
MAE:  0.3933
R2:   0.8096

Poly2
RMSE: 0.4991
MAE:  0.3566
R2:   0.8206

Poly4
RMSE: 47.0902
MAE:  6.9141
R2:   -2417.1671

LASSO
RMSE: 1.0463
MAE:  0.8517
R2:   0.2221


In [37]:
# Ridge reg. model with > 5 predictors, with CV tuning
ridge_model = Pipeline([("scaler", StandardScaler()),
    ("ridge", Ridge())])

param_grid = {"ridge__alpha": [0.01, 0.1, 1.0, 10.0, 100.0]}

#Ridge is easier to train on 1 scoring stat - will score the real CV on all
ridge_model = GridSearchCV(ridge_model,
    param_grid,
    cv=10,
    scoring="neg_root_mean_squared_error")

ridge_scores = cross_validate(ridge_model,
    x_train6,
    y_train,
    cv=5,
    scoring=scoring)

ridge_results = {"RMSE": -np.mean(ridge_scores["test_rmse"]),
    "MAE": -np.mean(ridge_scores["test_mae"]),
    "R2":  np.mean(ridge_scores["test_r2"])}

results["Ridge"] = ridge_results

# Print scores
for name, metrics in results.items():
    print(f"\n{name}")
    print(f"RMSE: {metrics['RMSE']:.4f}")
    print(f"MAE:  {metrics['MAE']:.4f}")
    print(f"R2:   {metrics['R2']:.4f}")


Interaction
RMSE: 0.5160
MAE:  0.3714
R2:   0.8094

Standard
RMSE: 0.5175
MAE:  0.3933
R2:   0.8096

Poly2
RMSE: 0.4991
MAE:  0.3566
R2:   0.8206

Poly4
RMSE: 47.0902
MAE:  6.9141
R2:   -2417.1671

LASSO
RMSE: 1.0463
MAE:  0.8517
R2:   0.2221

Ridge
RMSE: 1.0461
MAE:  0.8507
R2:   0.2223


In [39]:
# Elastic Net model with > 5 predictors, with CV tuning
elastic_model = Pipeline([("scaler", StandardScaler()),
    ("elastic", ElasticNet(max_iter=10000))])

param_grid = {"elastic__alpha": [0.01, 0.1, 1.0, 10.0],
    "elastic__l1_ratio": [0.2, 0.5, 0.8]}

elastic_model = GridSearchCV(elastic_model,
    param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error")

elastic_scores = cross_validate(elastic_model,
    x_train6,
    y_train,
    cv=5,
    scoring=scoring)

elastic_results = { "RMSE": -np.mean(elastic_scores["test_rmse"]),
    "MAE": -np.mean(elastic_scores["test_mae"]),
    "R2":  np.mean(elastic_scores["test_r2"])}

results["Elastic Net"] = elastic_results

# Print scores
for name, metrics in results.items():
    print(f"\n{name}")
    print(f"RMSE: {metrics['RMSE']:.4f}")
    print(f"MAE:  {metrics['MAE']:.4f}")
    print(f"R2:   {metrics['R2']:.4f}")


Interaction
RMSE: 0.5160
MAE:  0.3714
R2:   0.8094

Standard
RMSE: 0.5175
MAE:  0.3933
R2:   0.8096

Poly2
RMSE: 0.4991
MAE:  0.3566
R2:   0.8206

Poly4
RMSE: 47.0902
MAE:  6.9141
R2:   -2417.1671

LASSO
RMSE: 1.0463
MAE:  0.8517
R2:   0.2221

Ridge
RMSE: 1.0461
MAE:  0.8507
R2:   0.2223

Elastic Net
RMSE: 1.0461
MAE:  0.8515
R2:   0.2224


In [41]:
# Compare all 4 models on the test data
# Fit models
# model3 = Poly2
model3.fit(x_train, y_train)
lasso_model.fit(x_train6, y_train)
ridge_model.fit(x_train6, y_train)
elastic_model.fit(x_train6, y_train)

# Predict on test
y_pred_poly2 = model3.predict(x_test)
y_pred_lasso = lasso_model.predict(x_test6)
y_pred_ridge = ridge_model.predict(x_test6)
y_pred_elastic = elastic_model.predict(x_test6)

# Print metrics (same as the model dev)
test_results = {}

test_results["Poly2"] = {"RMSE": np.sqrt(mean_squared_error(y_test, y_pred_poly2)),
    "MAE": mean_absolute_error(y_test, y_pred_poly2),
    "R2": r2_score(y_test, y_pred_poly2)}

test_results["LASSO"] = {"RMSE": np.sqrt(mean_squared_error(y_test, y_pred_lasso)),
    "MAE": mean_absolute_error(y_test, y_pred_lasso),
    "R2": r2_score(y_test, y_pred_lasso)}

test_results["Ridge"] = {"RMSE": np.sqrt(mean_squared_error(y_test, y_pred_ridge)),
    "MAE": mean_absolute_error(y_test, y_pred_ridge),
    "R2": r2_score(y_test, y_pred_ridge)}

test_results["Elastic Net"] = {"RMSE": np.sqrt(mean_squared_error(y_test, y_pred_elastic)),
    "MAE": mean_absolute_error(y_test, y_pred_elastic),
    "R2": r2_score(y_test, y_pred_elastic)}

for name, metrics in test_results.items():
    print(f"\n{name}")
    print(f"RMSE: {metrics['RMSE']:.4f}")
    print(f"MAE:  {metrics['MAE']:.4f}")
    print(f"R2:   {metrics['R2']:.4f}")


Poly2
RMSE: 0.4724
MAE:  0.3411
R2:   0.8436

LASSO
RMSE: 1.0458
MAE:  0.8500
R2:   0.2337

Ridge
RMSE: 1.0452
MAE:  0.8486
R2:   0.2346

Elastic Net
RMSE: 1.0457
MAE:  0.8499
R2:   0.2338


Similar to the training stats, the 2-degree polynomial regression is the best model by far. Interestingly, the stats are better in the testing than the training for the `Poly2` model. This could be a result of a small CV, perhaps weighted by a bad model.

Now we will do a classification model test using log regression models to predict `wine_type`.